# Wstępna obróbka danych - częśc Filipa do sprawdzenia

In [ ]:
#prawdopodobnie to wszystkie biblio jakich potrzeba pamietajcie instalowac je przez terminal przez pip
import pandas as pd
import numpy as np
import matplotlib.pyplot as py
import scipy as sp
import seaborn as sb
import sklearn as sk

In [ ]:
#importujemy dane z pliku jako pd
df_apartments = pd.read_csv("data/apartments_for_rent_classified_100K.csv", sep=';', encoding="cp1250")
#df_apartments = pd.read_csv("/home/filip/Dokumenty/Python nauka/Kurs_Analiza/data/apartments_for_rent_classified_100K.csv", sep=';', encoding="cp1250")
#przy okazji sprawdzamy czy dobrze zaimportowało dane
df_apartments.head(5)

/tmp/ipykernel_24991/2193827058.py:2: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_apartments = pd.read_csv("data/apartments_for_rent_classified_100K.csv", sep=';', encoding="cp1250")


,id,category,title,amenities,bathrooms,bedrooms,currency,fee,has_photo,pets_allowed,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668640009,housing/rent/apartment,One BR 507 & 509 Esplanade,NaN,1.0,1.0,USD,No,Thumbnail,Cats,...,"$2,195",Monthly,542,507 509 Esplanade,Redondo Beach,CA,33.8520,-118.3759,RentLingo,1577360355
1,5668639818,housing/rent/apartment,Three BR 146 Lochview Drive,NaN,1.5,3.0,USD,No,Thumbnail,"Cats,Dogs",...,"$1,250",Monthly,1500,146 Lochview Dr,Newport News,VA,37.0867,-76.4941,RentLingo,1577360340
2,5668639686,housing/rent/apartment,Three BR 3101 Morningside Drive,NaN,2.0,3.0,USD,No,Thumbnail,NaN,...,"$1,395",Monthly,1650,3101 Morningside Dr,Raleigh,NC,35.8230,-78.6438,RentLingo,1577360332
3,5668639659,housing/rent/apartment,Two BR 209 Aegean Way,NaN,1.0,2.0,USD,No,Thumbnail,"Cats,Dogs",...,"$1,600",Monthly,820,209 Aegean Way,Vacaville,CA,38.3622,-121.9712,RentLingo,1577360330
4,5668639374,housing/rent/apartment,One BR 4805 Marquette NE,NaN,1.0,1.0,USD,No,Thumbnail,"Cats,Dogs",...,$975,Monthly,624,4805 Marquette NE,Albuquerque,NM,35.1038,-106.6110,RentLingo,1577360308


In [ ]:
df_apartments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 21 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id             100000 non-null  int64  
 1   category       100000 non-null  object 
 2   title          100000 non-null  object 
 3   amenities      83909 non-null   object 
 4   bathrooms      99937 non-null   float64
 5   bedrooms       99876 non-null   float64
 6   currency       100000 non-null  object 
 7   fee            100000 non-null  object 
 8   has_photo      100000 non-null  object 
 9   pets_allowed   39256 non-null   object 
 10  price          99999 non-null   float64
 11  price_display  99999 non-null   object 
 12  price_type     100000 non-null  object 
 13  square_feet    100000 non-null  int64  
 14  address        7943 non-null    object 
 15  cityname       99698 non-null   object 
 16  state          99698 non-null   object 
 17  latitude       99975 non-null 

### Krótka anliza
widzimy że brakujące dane są w kategoriach takich jak zwieżęta, pomieszczenia, adresy, nazwy miast, stany i wspolrzedne goegraficzne
poniżej zrobie zestawienie wyników i na ich bazie dokona się podziału na informację do odrzucenia i ewentualnego uzupełnienia

In [ ]:
df_apartments.isnull().sum()


id                   0
category             0
title                0
amenities        16091
bathrooms           63
bedrooms           124
currency             0
fee                  0
has_photo            0
pets_allowed     60744
price                1
price_display        1
price_type           0
square_feet          0
address          92057
cityname           302
state              302
latitude            25
longitude           25
source               0
time                 0
dtype: int64

In [ ]:
df_apartments['pets_allowed'].value_counts()

pets_allowed
Cats,Dogs         37279
Cats               1849
Dogs                127
Cats,Dogs,None        1
Name: count, dtype: int64

Z tego wynika, że brak danych (NA) dla kolumny pets należy potraktować jako not allowed, przydało by się też zmienić "Cats, Dogs, None" bedącą pojedynczym wpisem na Cats, Dogs

In [ ]:
# Zgodnie z powyższą sugestią dopisuję zmianę null na "None"
df_apartments["pets_allowed"] = df_apartments["pets_allowed"].fillna("None")

#zamiana pozycji jednostkowej Cats,Dogs,None na Cats,Dogs
df_apartments['pets_allowed'] = df_apartments['pets_allowed'].str.replace("Cats,Dogs,None", "Cats,Dogs", regex=False)
df_apartments['pets_allowed'].value_counts()

pets_allowed
Cats,Dogs    37280
Cats          1849
Dogs           127
Name: count, dtype: int64

In [ ]:
# Zgodnie ze wstępną analizą w analiza.ipynb usuwamy linię bez ceny
df_apartments = df_apartments.dropna(subset=["price"])

In [ ]:
df_apartments['amenities'].value_counts()

amenities
Parking                                                                                                                                       6210
Parking,Storage                                                                                                                               2119
Gym,Pool                                                                                                                                      1876
Pool                                                                                                                                          1490
Gym,Parking,Pool                                                                                                                              1195
                                                                                                                                              ... 
AC,Basketball,Cable or Satellite,Dishwasher,Gym,Internet Access,Parking,Patio/Deck,Playground,Pool,Refrigera

Tu mamy jeden wielki rozgardziasz, nie moze tak być że mamy aż 9851 pozycji tylko dlatego ze ktos coś traktuje jako udogodnienie a inny właściciel to uważa za podstawową funkcję, proponuję zrobić tak, rozbić kazdą komurkę w tej kolumnie oddzieloną przecinkiem na poszczególne udogodnienia, sprawdzić ilośc występujących udogodnień i na bazie tego i własnych odczuć zdefiniować które są rzeczywiście udogodnieniami a które nie
### Ps
okazuj się ze wszystkie udogodnienia mozna wziąc pod uwage, tylo trzeba stworzyć macierz binarną jak ponizej

In [ ]:
#tworzymy dla kazdego udogodnienia kolumne 0-1 gdzie 0 to nie wystepuje 1 wytepuje 
amenities_rozbite = amenities_clean.str.get_dummies(sep=',')
amenities_rozbite.sum()



AC                    15882
Alarm                   364
Basketball             4168
Cable or Satellite    12579
Clubhouse             19195
Dishwasher            16687
Doorman                 221
Elevator               4348
Fireplace             14986
Garbage Disposal       3899
Gated                  8687
Golf                     27
Gym                   37511
Hot Tub                4011
Internet Access       11127
Luxury                  211
Parking               44049
Patio/Deck            26650
Playground            11390
Pool                  43751
Refrigerator          14988
Storage               21731
TV                     4516
Tennis                 8542
View                   2101
Washer Dryer          26137
Wood Floors            8934
dtype: int64

Sprawdzamy tu zgodnosc stanow i miast bo to wazne dane do statystyki ceny

In [ ]:
# Filtrujemy wiersze, gdzie ZARÓWNO cityname JAK I state są puste
pokrycie_miast=df_apartments[df_apartments['cityname'].isnull() & df_apartments['state'].isnull()]
len(pokrycie_miast)


302

Wiemy w takim razie ze mamy 302 brakujace informacje jednoczesnie o stanie i o miescie ale mozemy sprawdzić czy nie mamy moze czasem ich polozenia geograficznego ale najpierw sprawdzmy zgodnosc szerokosci i wysokosci a raczej braku informacji o nich

In [ ]:
pokrycie_wspolrzednych=df_apartments[df_apartments['latitude'].isnull() & df_apartments['longitude'].isnull()]
len(pokrycie_wspolrzednych)

25

Mammy pełną zgodność co do braku danych czas na sprawdzenie czy mozemy wyznaczyć w pełni położenie miast za pomocą szerokości i wysokości geograficznej warunek jest taki że funkcja len musi pokazać 0 w innym wypadku nie możemy w pełni wyznaczyć nazw miast

In [ ]:
pokrycie=df_apartments[df_apartments['latitude'].isnull() & df_apartments['state'].isnull()]
len(pokrycie)

25

Mamy informacje że 25 miast ma nieznany stan i wyskosc wiec mamy 0 informacji o lokacji, mozna sprawdzic jeszcze adres jesli wyjdzie 0 oznacza to że mamy pełne informację

In [ ]:
pokrycie_adresu=df_apartments[df_apartments['address'].isnull() & df_apartments['latitude'].isnull()]
len(pokrycie_adresu)

25

Mamy 25 cen nieruchomości bez żadnych informacji o lokalizacji można te pozycje usunąc, za pomocą geopy mozna szukac pozostałych 277 pozycji pytanie czy warto bo przy 100k rekordow to tylko 0,3% zbioru

In [ ]:
#poniżej kod do pythona do usunięcia na razie jako komentarz
#df_apartments = df_apartments.dropna(subset=['cityname'])